# Create Hosted Agent From Image — `@azure/ai-projects`

This notebook demonstrates creating a Hosted Agent version from a container image, polling until it is active, routing the agent endpoint to that version, and then invoking it via the OpenAI Responses API.

It mirrors the [`createHostedAgentFromImage.ts`](./createHostedAgentFromImage.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` (and the `az` CLI used by the RBAC cell) can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_AGENT_CONTAINER_IMAGE`, `FOUNDRY_HOSTED_AGENT_NAME` (optional; defaults to `MyHostedAgentForNotebook`), `FOUNDRY_SUBSCRIPTION_ID` (optional; falls back to the active `az` subscription).

**Note on RBAC:** the hosted agent's instance identity must have the **Foundry User** role on the Foundry account for the container image version to provision; otherwise the poll cell reports `status: failed`. The RBAC cell assigns this role via the `az` CLI. Set `SKIP_RBAC=true` if the assignment is managed out-of-band (newer Foundry accounts may not require it).

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  AgentEndpointConfig,
  HostedAgentDefinition,
  ProtocolVersionRecord,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const agentName = process.env["FOUNDRY_HOSTED_AGENT_NAME"] ?? "MyHostedAgentForNotebook";
const image = process.env["FOUNDRY_AGENT_CONTAINER_IMAGE"] ?? "<agent image>";

console.log(`Agent: ${agentName}`);
console.log(`Image: ${image}`);

Agent: MyHostedAgent9
Image: crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest
Image: crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest


In [2]:
// Create the AI Project client
// Annotated as `any` so tslab does not try to emit non-portable declarations
// referencing the deep `node_modules/openai` (pnpm junction) path.
const project: any = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Create a hosted agent version from a container image
const created = await project.agents.createVersion(
  agentName,
  {
    kind: "hosted",
    cpu: "0.5",
    memory: "1Gi",
    container_configuration: { image: image } as AgentEndpointConfig,
    protocol_versions: [{ protocol: "responses", version: "1.0.0" } as ProtocolVersionRecord],
  } as HostedAgentDefinition,
  {
    metadata: { enableVnextExperience: "true" },
  },
);
console.log(`Created hosted agent version: ${created.version}`);

Created hosted agent version: 108


In [ ]:
// Ensure the hosted agent's instance identity has the "Foundry User" role on
// the Foundry account, so the version can provision (pull the image, access the
// account). Mirrors `ensure_agent_identity_rbac` from the Python samples.
// Uses the `az` CLI (same `az login` identity already required) — no extra deps.
// The `az` executable differs by platform (`az.cmd` on Windows) and is invoked
// without a shell so environment-derived values cannot be interpreted as shell
// syntax. Set SKIP_RBAC=true to skip if the assignment is managed out-of-band.
import { execFileSync } from "node:child_process";

const FOUNDRY_USER_ROLE_ID = "53ca6127-db72-4b80-b1b0-d745d6d5456d";
const azExe = process.platform === "win32" ? "az.cmd" : "az";

const az = (args: string[]): string =>
  execFileSync(azExe, args, { encoding: "utf8" });

if ((process.env["SKIP_RBAC"] ?? "").toLowerCase() === "true") {
  console.log("Skipping RBAC setup (SKIP_RBAC=true).");
} else {
  const principalId = created.instance_identity?.principal_id;
  if (!principalId) {
    throw new Error("Agent instance_identity.principal_id is not available.");
  }

  // Parse account name from the endpoint:
  // https://<account>.services.ai.azure.com/api/projects/<project>
  const accountName = new URL(projectEndpoint).hostname.split(".")[0];

  const subscriptionId =
    process.env["FOUNDRY_SUBSCRIPTION_ID"] ??
    JSON.parse(az(["account", "show", "-o", "json"])).id;

  // Resolve the Foundry (CognitiveServices) account resource ID for the scope.
  const accounts = JSON.parse(
    az([
      "resource",
      "list",
      "--resource-type",
      "Microsoft.CognitiveServices/accounts",
      "--name",
      accountName,
      "--subscription",
      subscriptionId,
      "-o",
      "json",
    ]),
  ) as Array<{ id: string; name: string }>;
  const scope = accounts.find((a) => a.name === accountName)?.id;
  if (!scope) {
    throw new Error(`Could not locate Azure AI account '${accountName}' in subscription.`);
  }

  // Only an already-existing assignment is safe to ignore; anything else
  // (403, invalid scope, transient CLI failure) means provisioning would fail
  // silently, so rethrow it.
  const roleAssignmentExists = (): boolean => {
    const existing = JSON.parse(
      az([
        "role",
        "assignment",
        "list",
        "--assignee",
        principalId,
        "--role",
        FOUNDRY_USER_ROLE_ID,
        "--scope",
        scope,
        "-o",
        "json",
      ]),
    ) as unknown[];
    return existing.length > 0;
  };

  if (roleAssignmentExists()) {
    console.log(`Foundry User role already assigned to principal ${principalId}.`);
  } else {
    az([
      "role",
      "assignment",
      "create",
      "--assignee-object-id",
      principalId,
      "--assignee-principal-type",
      "ServicePrincipal",
      "--role",
      FOUNDRY_USER_ROLE_ID,
      "--scope",
      scope,
    ]);
    console.log(`Assigned Foundry User role to principal ${principalId}.`);
  }
}

In [7]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, created.version);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: active (attempt 1/60)


In [8]:
// Patch the agent endpoint to route to the new version
const endpointConfig: AgentEndpointConfig = {
  version_selector: {
    version_selection_rules: [
      {
        type: "FixedRatio",
        agent_version: created.version,
        traffic_percentage: 100,
      },
    ],
  },
  protocol_configuration: { responses: {} },
};
await project.agents.patchAgentObject(agentName, { agentEndpoint: endpointConfig });
console.log(`Agent endpoint configured for version ${created.version}`);

const fetched = await project.agents.getVersion(agentName, created.version);
console.log(`Fetched hosted agent version: ${fetched.version}, status: ${fetched.status}`);

Agent endpoint configured for version 108
Fetched hosted agent version: 108, status: active


In [10]:
// Invoke the agent via the OpenAI Responses API
// Annotated as `any` so tslab does not emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient({
  azureConfig: { allowPreview: true, agentName: agentName },
});

const userInput = "Good morning!";
const response = await openAIClient.responses.create({ input: userInput });
console.log(`Sent: ${userInput}`);
console.log(`Response output: ${response.output_text}`);

Sent: Good morning!
Response output: Echo: Good morning!
Response output: Echo: Good morning!
